In [24]:
import pandas as pd
import re
from collections import defaultdict
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

df = pd.DataFrame(data, columns=["text", "class"])

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text.split()

df["cleaned_doc"] = df["text"].apply(preprocess)




In [25]:
# Build vocabulary and class word counts

class_word_counts = defaultdict(lambda: defaultdict(int))
class_counts = defaultdict(int)
vocab = set()

for _, row in df.iterrows():
    label = row["class"]
    class_counts[label] += 1
    
    for word in row["cleaned_doc"]:
        class_word_counts[label][word] += 1
        vocab.add(word)

vocab_size = len(vocab)

print("Class counts:", dict(class_counts))
print("Vocab size:", vocab_size)


Class counts: {'SPAM': 5, 'HAM': 6}
Vocab size: 44


In [26]:
vocab_size = len(vocab)
print("Vocab size:", vocab_size)

print("=== BAG OF WORDS ===\n")

for label in class_word_counts:
    print(f"{label} WORD FREQUENCIES:\n")

    word_counts = class_word_counts[label]
    total_words = sum(word_counts.values())

    for word, count in sorted(word_counts.items()):
        print(f"{word:12} : {count}")

    print("\nTotal words:", total_words)
    print("-" * 40)


Vocab size: 44
=== BAG OF WORDS ===

SPAM WORD FREQUENCIES:

50           : 1
a            : 1
click        : 1
for          : 2
free         : 2
get          : 1
here         : 1
iphone       : 1
limited      : 1
lowest       : 1
meds         : 1
money        : 1
now          : 1
off          : 1
price        : 1
prizes       : 1
time         : 1
today        : 1
win          : 1
your         : 1

Total words: 22
----------------------------------------
HAM WORD FREQUENCIES:

3            : 1
are          : 2
at           : 2
can          : 1
catch        : 1
dinner       : 1
for          : 1
hi           : 1
how          : 1
in           : 1
lets         : 1
meeting      : 2
mom          : 1
office       : 2
on           : 1
pm           : 1
report       : 1
send         : 1
still        : 1
team         : 1
the          : 3
tomorrow     : 2
up           : 1
we           : 1
you          : 2

Total words: 33
----------------------------------------


In [27]:
total_docs = sum(class_counts.values())

priors = {
    label: class_counts[label] / total_docs
    for label in class_counts
}

print("Priors:", priors)

Priors: {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}


In [28]:
def get_likelihood(word, label, alpha=1):
    numerator = class_word_counts[label][word] + alpha
    denominator = sum(class_word_counts[label].values()) + alpha * vocab_size
    return numerator / denominator

In [29]:
def predict_manual(sentence):
    words = preprocess(sentence)
    
    print(f"\n--- Analyzing: '{sentence}' ---")

    scores = {}

    for label in priors:
        score = priors[label]
        print(f"\n  Class: {label}")
        print(f"    Prior: {score:.4f}")

        for word in words:
            prob = get_likelihood(word, label)
            score *= prob
            print(f"    Word: '{word}' -> {prob:.4f}")

        scores[label] = score
        print(f"    Total Score: {score:.8f}")

    predicted_class = max(scores, key=scores.get)
    print(f"\n>> PREDICTION: {predicted_class}")
    return predicted_class


test_sentences = [
    "Limited offer, click here!", 
    "Meeting at 2 PM with the manager."
]

print("\n--- Manual Predictions with Probabilities ---")
for sent in test_sentences:
    predict_manual(sent)



--- Manual Predictions with Probabilities ---

--- Analyzing: 'Limited offer, click here!' ---

  Class: SPAM
    Prior: 0.4545
    Word: 'limited' -> 0.0303
    Word: 'offer' -> 0.0152
    Word: 'click' -> 0.0303
    Word: 'here' -> 0.0303
    Total Score: 0.00000019

  Class: HAM
    Prior: 0.5455
    Word: 'limited' -> 0.0130
    Word: 'offer' -> 0.0130
    Word: 'click' -> 0.0130
    Word: 'here' -> 0.0130
    Total Score: 0.00000002

>> PREDICTION: SPAM

--- Analyzing: 'Meeting at 2 PM with the manager.' ---

  Class: SPAM
    Prior: 0.4545
    Word: 'meeting' -> 0.0152
    Word: 'at' -> 0.0152
    Word: '2' -> 0.0152
    Word: 'pm' -> 0.0152
    Word: 'with' -> 0.0152
    Word: 'the' -> 0.0152
    Word: 'manager' -> 0.0152
    Total Score: 0.00000000

  Class: HAM
    Prior: 0.5455
    Word: 'meeting' -> 0.0390
    Word: 'at' -> 0.0390
    Word: '2' -> 0.0130
    Word: 'pm' -> 0.0260
    Word: 'with' -> 0.0130
    Word: 'the' -> 0.0519
    Word: 'manager' -> 0.0130
    Total Sco

In [30]:
X_train = df['text']
y_train = df['class']

vectorizer = CountVectorizer() 
X_train_vec = vectorizer.fit_transform(X_train)

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

test_sentences = [
    "Limited offer, click here!", 
    "Meeting at 2 PM with the manager."
]

X_test_vec = vectorizer.transform(test_sentences)

predictions = clf.predict(X_test_vec)

print("Scikit-Learn Predictions")
for sent, pred in zip(test_sentences, predictions):
    print(f"'{sent}' -> {pred}")

Scikit-Learn Predictions
'Limited offer, click here!' -> SPAM
'Meeting at 2 PM with the manager.' -> HAM
